In [36]:
import sys
sys.path.append("..")

from pathlib import Path
import pandas as pd
from turbofan.data import load_cmapss

In [37]:
RAW = Path("../data/raw")
path = Path(RAW / "test_FD001.txt")

In [38]:
test = load_cmapss(path)
test.sample(2)

,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
10418,81,31,-0.0023,0.0001,100.0,518.67,642.41,1591.13,1401.66,14.62,21.61,554.06,2387.98,9071.54,1.3,47.37,522.01,2387.95,8145.79,8.4054,0.03,393,2388,100.0,38.97,23.3846
1051,10,156,0.0003,0.0003,100.0,518.67,642.20,1588.48,1400.95,14.62,21.61,554.45,2388.07,9056.63,1.3,47.13,521.14,2388.08,8144.46,8.4179,0.03,391,2388,100.0,38.92,23.3511


In [39]:
test.shape

(13096, 26)

In [40]:
test_life = test.groupby('unit')["cycle"].max()

In [41]:
test_life.describe()

count    100.000000
mean     130.960000
std       53.593479
min       31.000000
25%       88.750000
50%      133.500000
75%      164.250000
max      303.000000
Name: cycle, dtype: float64

In [42]:
rul = pd.read_csv(RAW / "RUL_FD001.txt", sep=r"\s+", header=None, names=["rul"])

In [43]:
rul

,rul
0,112
1,98
2,69
3,82
4,91
...,...
95,137
96,82
97,59
98,117


In [44]:
print(rul.shape)
print(rul.head)

(100, 1)
<bound method NDFrame.head of     rul
0   112
1    98
2    69
3    82
4    91
..  ...
95  137
96   82
97   59
98  117
99   20

[100 rows x 1 columns]>


In [45]:
rul["unit"] = range(1, len(rul) + 1)
rul.head

<bound method NDFrame.head of     rul  unit
0   112     1
1    98     2
2    69     3
3    82     4
4    91     5
..  ...   ...
95  137    96
96   82    97
97   59    98
98  117    99
99   20   100

[100 rows x 2 columns]>

In [46]:
from turbofan.features import add_rolling_features

test_feat = add_rolling_features(test)
last_rows = test_feat.groupby("unit").tail(1)

In [47]:
print(last_rows.shape)
print(last_rows[["unit", "cycle", "sensor_11", "sensor_11_mean"]].head())

(100, 56)
     unit  cycle  sensor_11  sensor_11_mean
30      1     31      47.23          47.288
79      2     49      47.67          47.509
205     3    126      47.88          47.705
311     4    106      47.65          47.630
409     5     98      47.46          47.533


In [48]:
test_final = last_rows.merge(rul, on="unit")
print(test_final[["unit", "cycle", "rul"]].head())

   unit  cycle  rul
0     1     31  112
1     2     49   98
2     3    126   69
3     4    106   82
4     5     98   91


In [49]:
print(test_final.shape)

(100, 57)


In [50]:
test_final.head()

,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_2_mean,sensor_2_std,sensor_3_mean,sensor_3_std,sensor_4_mean,sensor_4_std,sensor_6_mean,sensor_6_std,sensor_7_mean,sensor_7_std,sensor_8_mean,sensor_8_std,sensor_9_mean,sensor_9_std,sensor_11_mean,sensor_11_std,sensor_12_mean,sensor_12_std,sensor_13_mean,sensor_13_std,sensor_14_mean,sensor_14_std,sensor_15_mean,sensor_15_std,sensor_17_mean,sensor_17_std,sensor_20_mean,sensor_20_std,sensor_21_mean,sensor_21_std,rul
0,1,31,-0.0006,0.0004,100.0,518.67,642.58,1581.22,1398.91,14.62,21.61,554.42,2388.08,9056.40,1.3,47.23,521.79,2388.06,8130.11,8.4024,0.03,393,2388,100.0,38.81,23.3552,642.295,0.282105,1586.528,4.612331,1400.719,3.465444,21.61,0.0,554.070,0.566117,2388.071,0.030714,9049.068,4.850033,47.288,0.111335,521.930,0.273942,2388.069,0.028067,8130.849,2.631744,8.42013,0.022340,392.2,0.788811,38.933,0.114120,23.36944,0.026390,112
1,2,49,0.0018,-0.0001,100.0,518.67,642.55,1586.59,1410.83,14.62,21.61,553.52,2388.10,9044.77,1.3,47.67,521.74,2388.09,8126.90,8.4505,0.03,391,2388,100.0,38.81,23.2618,642.712,0.126912,1589.460,2.882819,1407.271,3.472279,21.61,0.0,553.640,0.346731,2388.110,0.036209,9047.351,3.331011,47.509,0.088248,521.514,0.387562,2388.115,0.019579,8127.584,2.054952,8.44631,0.023850,392.4,0.843274,38.847,0.084859,23.29325,0.049705,98
2,3,126,-0.0016,0.0004,100.0,518.67,642.88,1589.75,1418.89,14.62,21.61,552.59,2388.16,9049.26,1.3,47.88,520.83,2388.14,8131.46,8.4119,0.03,395,2388,100.0,38.93,23.2740,642.854,0.264500,1593.653,3.906456,1413.986,3.114829,21.61,0.0,552.823,0.355529,2388.162,0.026162,9048.908,3.412294,47.705,0.131930,520.795,0.301598,2388.161,0.027264,8129.387,2.904323,8.46210,0.022946,393.9,0.994429,38.743,0.134002,23.22671,0.039177,69
3,4,106,0.0012,0.0004,100.0,518.67,642.78,1594.53,1406.88,14.62,21.61,552.64,2388.13,9051.30,1.3,47.65,521.88,2388.11,8133.64,8.4634,0.03,395,2388,100.0,38.58,23.2581,642.838,0.189491,1591.346,2.855588,1411.322,4.266794,21.61,0.0,553.344,0.441316,2388.136,0.044771,9055.564,5.072721,47.630,0.051854,521.324,0.298113,2388.113,0.024967,8133.675,2.720683,8.45089,0.016581,393.4,1.074968,38.783,0.102637,23.26245,0.054676,82
4,5,98,-0.0013,-0.0004,100.0,518.67,642.27,1589.94,1419.36,14.62,21.61,553.29,2388.10,9053.99,1.3,47.46,521.00,2388.15,8125.74,8.4362,0.03,394,2388,100.0,38.75,23.4117,642.722,0.203076,1590.282,2.914213,1409.859,4.969282,21.61,0.0,553.430,0.468828,2388.120,0.029814,9051.898,3.535615,47.533,0.114993,521.115,0.239270,2388.123,0.032677,8130.344,3.669963,8.45198,0.018150,393.5,0.707107,38.774,0.082219,23.25765,0.078644,91


Loading the trained model back

In [51]:
import json
import joblib
from turbofan.data import load_test_rul
from turbofan.features import last_cycle_per_engine

ImportError: cannot import name 'load_test_rul' from 'turbofan.data' (c:\dev\predictive-maintenance-mlops\notebooks\..\turbofan\data.py)